# DoE Scientific Analyzer Example

This notebook demonstrates a practical Design of Experiments workflow for chemical/process optimization. We load a reaction dataset, fit a quadratic response-surface model, rank factor effects, generate plots, run PCA, and predict optimal process settings.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.doe_analysis import coefficient_table, fit_doe_model, model_metrics, optimize_response, run_pca
from src.utils import load_dataset
from src.visualization import (
    plot_observed_vs_predicted,
    plot_pareto,
    plot_pca_loadings,
    plot_pca_scores,
    plot_response_surface,
    save_figure,
)

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load the Dataset

The sample file contains three controllable process factors and three measured responses. Each row represents one experimental run.

In [ ]:
df = load_dataset(PROJECT_ROOT / 'examples' / 'sample_dataset.csv')
df.head()

## 2. Define Factors and Response

Here we optimize reaction yield as a function of temperature, catalyst loading, and reaction time.

In [ ]:
factors = ['temperature_c', 'catalyst_pct', 'time_min']
response = 'yield_pct'

## 3. Fit a Quadratic Response-Surface Model

A quadratic model estimates main effects, two-factor interactions, and curvature. This is commonly used in Response Surface Methodology after screening or central-composite style experiments.

In [ ]:
model = fit_doe_model(df, factors=factors, response=response, model_type='quadratic')
model.formula

In [ ]:
model_metrics(model)

## 4. Rank Factor Effects

The coefficient table ranks terms by absolute t statistic. Large values indicate terms with stronger evidence of an effect on the response.

In [ ]:
effects = coefficient_table(model)
effects

In [ ]:
fig = plot_pareto(effects)
save_figure(fig, OUTPUT_DIR / 'notebook_pareto_chart.png')

## 5. Model Diagnostics

Observed vs predicted plots are a quick check for lack of fit, outliers, or systematic prediction bias.

In [ ]:
fig = plot_observed_vs_predicted(model)
save_figure(fig, OUTPUT_DIR / 'notebook_observed_vs_predicted.png')

## 6. Response Surface Methodology

A contour plot shows how the predicted response changes across two factors while other factors are held at their center values.

In [ ]:
fig = plot_response_surface(model, 'temperature_c', 'catalyst_pct')
save_figure(fig, OUTPUT_DIR / 'notebook_response_surface.png')

## 7. PCA for Multivariate Exploration

PCA helps summarize relationships across factors and responses. It is especially useful when several quality attributes must be understood together.

In [ ]:
pca_result = run_pca(df, factors + ['yield_pct', 'purity_pct', 'impurity_pct'])
pca_result['explained_variance'][:3]

In [ ]:
save_figure(plot_pca_scores(pca_result), OUTPUT_DIR / 'notebook_pca_scores.png')
save_figure(plot_pca_loadings(pca_result), OUTPUT_DIR / 'notebook_pca_loadings.png')

## 8. Predict Optimal Parameters

The optimizer searches within the tested experimental region. Treat the predicted optimum as a candidate for confirmatory experiments.

In [ ]:
best_parameters, predicted_response = optimize_response(model, goal='maximize')
best_parameters, predicted_response